# Product Pulse — EDA & Business KPI Analysis

**Objective:** understand product-level risk signals before building the ML model.

This notebook covers:
- Data loading and quality checks
- Business KPIs
- Product/category performance
- Return-rate trends
- Review/complaint signals
- Identification of risk signals
- Initial ML feature engineering


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
DATA = ROOT / "data" / "processed"

products = pd.read_csv(DATA / "products.csv")
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
returns = pd.read_csv(DATA / "returns.csv", parse_dates=["return_date"])
reviews = pd.read_csv(DATA / "reviews.csv", parse_dates=["review_date"])
tickets = pd.read_csv(DATA / "support_tickets.csv", parse_dates=["created_at"])
metrics = pd.read_csv(DATA / "product_metrics_monthly.csv", parse_dates=["month"])
labels = pd.read_csv(DATA / "product_risk_labels.csv", parse_dates=["month"])


In [ ]:
print("Shapes:")
for name, df in {
    "products": products,
    "orders": orders,
    "items": items,
    "returns": returns,
    "reviews": reviews,
    "tickets": tickets,
    "metrics": metrics,
    "labels": labels,
}.items():
    print(f"{name:10s}: {df.shape}")

print("\nMissing cells:")
print(metrics.isna().sum())


## Business KPIs

The initial dashboard/business view should focus on a small set of metrics rather than dozens of charts:

- Total orders / units sold
- Return rate
- Average rating
- Negative-review rate
- Support-ticket volume
- Products requiring attention


In [ ]:
revenue = (items["quantity"] * items["unit_price"]).sum()
return_rate = returns["return_id"].nunique() / items["order_item_id"].nunique()
avg_rating = reviews["rating"].mean()
negative_review_rate = (reviews["rating"] <= 3).mean()

kpis = pd.Series({
    "Revenue": revenue,
    "Orders": orders["order_id"].nunique(),
    "Units Sold": items["quantity"].sum(),
    "Return Rate": return_rate,
    "Average Rating": avg_rating,
    "Negative Review Rate": negative_review_rate,
    "Support Tickets": tickets["ticket_id"].nunique(),
})
kpis


## Monthly return-rate trend


In [ ]:
monthly = metrics.groupby("month", as_index=False).agg(
    units_sold=("units_sold","sum"),
    returned_units=("returned_units","sum")
)
monthly["return_rate"] = monthly["returned_units"] / monthly["units_sold"]

plt.figure(figsize=(10,5))
plt.plot(monthly["month"], monthly["return_rate"], marker="o")
plt.title("Overall Monthly Return Rate")
plt.xlabel("Month")
plt.ylabel("Return Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Product risk signals


In [ ]:
latest = metrics.loc[metrics.groupby("product_id")["month"].idxmax()].copy()
latest = latest.merge(
    products[["product_id","product_name","category","risk_profile"]],
    on="product_id",
    how="left"
)

latest["risk_signal_score"] = (
    0.45 * np.clip(latest["return_rate"]/0.20, 0, 1) +
    0.30 * np.clip((5-latest["avg_rating"])/2.5, 0, 1) +
    0.15 * np.clip(latest["negative_review_rate"]/0.65, 0, 1) +
    0.10 * np.clip(latest["support_ticket_count"]/30, 0, 1)
) * 100

latest.sort_values("risk_signal_score", ascending=False).head(10)


## Initial ML features

We will use historical signals to predict **next-month high risk**.

Important: the target month must always be in the future relative to the feature month to avoid data leakage.


In [ ]:
feature_df = metrics.sort_values(["product_id","month"]).copy()

feature_df["return_rate_3m_avg"] = (
    feature_df.groupby("product_id")["return_rate"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

feature_df["return_rate_change_1m"] = (
    feature_df.groupby("product_id")["return_rate"].diff().fillna(0)
)

feature_df["rating_change_1m"] = (
    feature_df.groupby("product_id")["avg_rating"].diff().fillna(0)
)

feature_df["negative_review_change_1m"] = (
    feature_df.groupby("product_id")["negative_review_rate"].diff().fillna(0)
)

feature_df["ticket_rate_per_100_orders"] = (
    feature_df["support_ticket_count"] /
    feature_df["orders"].replace(0, np.nan) * 100
).fillna(0)

feature_df.head()
